<a href="https://colab.research.google.com/github/dang710206/Baitap-AI-/blob/main/Bt15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx folium networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.4 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import networkx as nx
import folium
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import random

#Hệ thống phân bố xe cứu thương
place = "Phú Nhuận, Thành phố Hồ Chí Minh, Việt Nam"

G = ox.graph_from_place(place, network_type='drive', simplify=True, retain_all=True)
G = ox.truncate.largest_component(G, strongly=True)   # Tránh lỗi NoPath

G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

print(f"Mạng lưới đường bộ: {len(G.nodes())} nút, {len(G.edges())} cạnh\n")

np.random.seed(42)
# Tạo 100 điểm tai nạn ngẫu nhiên quanh khu vực Phú Nhuận
accidents = np.array([[10.80 + np.random.normal(0, 0.015),
                       106.67 + np.random.normal(0, 0.015)] for _ in range(100)])

# Phân cụm thành 3 vùng nguy cơ cao bằng KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(accidents)
centers = kmeans.cluster_centers_

print("Đã xác định 3 vùng nguy cơ cao tai nạn giao thông.")

ambulances = [
    [10.802, 106.675],
    [10.785, 106.705],
    [10.775, 106.698],
    [10.782, 106.702],
    [10.768, 106.688]
]

# Đề xuất vị trí tối ưu: xe gần cụm tai nạn nhất
proposed = []
for center in centers:
    dists = cdist([center], ambulances)[0]
    best_idx = np.argmin(dists)
    proposed.append(ambulances[best_idx])

center_map = [10.80, 106.67]
m = folium.Map(location=center_map, zoom_start=14, tiles="OpenStreetMap")

for center in centers:
    folium.Circle(location=center, radius=800,
                  color='red', fill=True, fill_opacity=0.25,
                  popup='Vùng nguy cơ cao tai nạn').add_to(m)

# Một số điểm tai nạn
for lat, lon in accidents[::8]:   # lấy thưa hơn để bản đồ sạch
    folium.CircleMarker([lat, lon], radius=3, color='orange', fill=True,
                        popup='Điểm tai nạn').add_to(m)

# Vị trí xe cứu thương hiện tại
for lat, lon in ambulances:
    folium.Marker([lat, lon],
                  popup='Xe cứu thương hiện tại',
                  icon=folium.Icon(color='blue', icon='ambulance', prefix='fa')).add_to(m)

# Vẽ vị trí đề xuất (xanh lá)
for lat, lon in proposed:
    folium.Marker([lat, lon],
                  popup='Vị trí đề xuất tối ưu',
                  icon=folium.Icon(color='green', icon='flag', prefix='fa')).add_to(m)

# Trung tâm cấp cứu
folium.Marker([10.78, 106.70],
              popup='Bệnh viện / Trung tâm cấp cứu',
              icon=folium.Icon(color='red', icon='plus', prefix='fa')).add_to(m)

# Thêm layer đường từ OSMnx
folium.TileLayer('cartodbpositron').add_to(m)  # nền sạch hơn

display(m)


=== BÀI 23.15: HỆ THỐNG PHÂN BỔ XE CỨU THƯƠNG THÔNG MINH DỰA TRÊN HOTSPOT TAI NẠN ===

Mạng lưới đường bộ: 333 nút, 710 cạnh

Đã xác định 3 vùng nguy cơ cao tai nạn giao thông.
